# **Лабораторная 2. Формирование отчётов в Apache Spark**


Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.


Получившийся отчёт сохранить в формате Apache Parquet.


Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.


Набор данных
Архивы сайтов Stack Exchange доступны по адресу https://archive.org/details/stackexchange.


В папке data данного репозитория вам доступны:


выборка данных posts_sample.xml (из stackoverflow.com-Posts.7z),
файл со списком языков programming-languages.csv, собранных с вики-страницы https://en.wikipedia.org/wiki/List_of_programming_languages.
Рекомендуется отлаживать решение на небольшой выборке данных posts_sample.xml.



In [ ]:
!pip install pyspark

Скачивание файлов

In [9]:
# Скачиваем список языков
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/bachelor/data/programming-languages.csv

# Скачиваем пример постов
!wget https://git.ai.ssau.ru/tk/big_data/raw/branch/bachelor/data/posts_sample.xml

print("Файлы успешно загружены в Colab!")

--2026-05-05 18:42:43--  https://git.ai.ssau.ru/tk/big_data/raw/branch/bachelor/data/programming-languages.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40269 (39K) [text/plain]
Saving to: ‘programming-languages.csv’

programming-languag 100%[===================>]  39.33K   129KB/s    in 0.3s    

2026-05-05 18:42:45 (129 KB/s) - ‘programming-languages.csv’ saved [40269/40269]

--2026-05-05 18:42:45--  https://git.ai.ssau.ru/tk/big_data/raw/branch/bachelor/data/posts_sample.xml
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74162295 (71M) [text/plain]
Saving to: ‘posts_sample.xml’

posts_sample.xml    100%[===================>]  70.73M   356KB/s    in 2m 13s  

2026-05-05 18:44:59 (544 KB/s

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import xml.etree.ElementTree as ET

spark = SparkSession.builder.master("local[*]").getOrCreate()

posts_path = "posts_sample.xml"
languages_path = "programming-languages.csv"

In [12]:
# Читаем файл как текст
raw_rdd = spark.sparkContext.textFile(posts_path)
indexed_rdd = raw_rdd.zipWithIndex()
total_count = indexed_rdd.count()

# Фильтруем: убираем заголовок XML (строки 0, 1) и последнюю строку
data_rdd = indexed_rdd.filter(lambda x: x[1] > 1 and x[1] < total_count - 1).map(lambda x: x[0])

# Функция парсинга атрибутов Tags и CreationDate
def parse_row(xml_row):
    try:
        root = ET.fromstring(xml_row.strip())
        return (root.attrib.get('Tags'), root.attrib.get('CreationDate'))
    except:
        return None

# Превращаем в таблицу DataFrame
parsed_posts = data_rdd.map(parse_row).filter(lambda x: x is not None and x[0] is not None)
posts_df = spark.createDataFrame(parsed_posts, ["tags", "creation_date"])

Обработка данных

In [13]:
# Загружаем список языков
languages_df = spark.read.csv(languages_path, header=True)
langs = [row['name'].lower() for row in languages_df.collect()]

# UDF для поиска языков в тегах
def extract_langs(tag_str):
    tag_str = tag_str.lower().replace('<', ' ').replace('>', ' ')
    return [l for l in langs if l in tag_str.split()]

extract_langs_udf = F.udf(extract_langs, "array<string>")

# Основная обработка
report = posts_df.withColumn("year", F.year(F.col("creation_date").cast("timestamp"))) \
    .filter((F.col("year") >= 2010) & (F.col("year") <= 2020)) \
    .withColumn("lang", F.explode(extract_langs_udf("tags"))) \
    .groupBy("year", "lang") \
    .count()

# Оконная функция для выбора ТОП-10
from pyspark.sql.window import Window
window_spec = Window.partitionBy("year").orderBy(F.desc("count"))

final_top10 = report.withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 10) \
    .drop("rank") \
    .orderBy("year", F.desc("count"))

# Вывод результата на экран
final_top10.show(100)

+----+-----------+-----+
|year|       lang|count|
+----+-----------+-----+
|2010|       java|   52|
|2010|        php|   46|
|2010| javascript|   44|
|2010|     python|   26|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   12|
|2010|     delphi|    8|
|2010|applescript|    3|
|2010|          r|    3|
|2011|        php|  102|
|2011|       java|   93|
|2011| javascript|   83|
|2011|     python|   37|
|2011|objective-c|   34|
|2011|          c|   24|
|2011|       ruby|   20|
|2011|       perl|    9|
|2011|     delphi|    8|
|2011|       bash|    7|
|2012|        php|  154|
|2012| javascript|  132|
|2012|       java|  124|
|2012|     python|   69|
|2012|objective-c|   45|
|2012|       ruby|   27|
|2012|          c|   27|
|2012|       bash|   10|
|2012|          r|    9|
|2012|      scala|    6|
|2013|        php|  198|
|2013| javascript|  198|
|2013|       java|  194|
|2013|     python|   90|
|2013|objective-c|   40|
|2013|          c|   36|
|2013|       ruby|   32|


Сохранение в Parquet

In [14]:
final_top10.write.mode("overwrite").parquet("top_10_languages_report.parquet")
print("Отчет успешно сохранен!")

Отчет успешно сохранен!
